In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, matthews_corrcoef,
                             precision_score, recall_score, roc_auc_score,
                             confusion_matrix, brier_score_loss, cohen_kappa_score, log_loss)
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Conv1D, Bidirectional, Reshape, Dropout

# ==========================================
# 1. MODEL DEFINITIONS
# ==========================================
def get_mlp(shape):
    ins = Input(shape=(shape,))
    x = Dense(128, activation='swish')(ins)
    x = Dropout(0.4)(x)
    x = Dense(64, activation='swish')(x)
    outs = Dense(1, activation='sigmoid')(x)
    model = Model(ins, outs); model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

def get_cnn_gru(shape):
    ins = Input(shape=(shape,))
    x = Reshape((shape, 1))(ins)
    x = Conv1D(64, kernel_size=3, padding='same', activation='relu')(x)
    x = Bidirectional(GRU(64))(x)
    outs = Dense(1, activation='sigmoid')(x)
    model = Model(ins, outs); model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

# ==========================================
# 2. CROSS-VALIDATION LOOP
# ==========================================
df = pd.read_csv('file1.csv')
X = df.drop(['UDI', 'Product ID', 'Date', 'Diagnostic'], axis=1, errors='ignore')
y = (df['Diagnostic'] != 'No failure').astype(int)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scal', StandardScaler())]), X.select_dtypes(include=['int64', 'float64']).columns.tolist()),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), X.select_dtypes(include=['object']).columns.tolist())
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
strategies = ['Soft Voting', 'Weighted Voting', 'Stacking']
metrics_list = ['acc', 'f1', 'prec', 'rec', 'auc', 'mcc', 'kappa', 'brier', 'logloss']

# Dictionary to store results for each fold
cv_results = {s: {m: [] for m in metrics_list} for s in strategies}

print("Starting 5-Fold Cross-Validation Comparison...")

for train_idx, val_idx in skf.split(X, y):
    Xt, Xv = X.iloc[train_idx], X.iloc[val_idx]
    yt, yv = y.iloc[train_idx], y.iloc[val_idx]

    Xt_p = preprocessor.fit_transform(Xt)
    Xv_p = preprocessor.transform(Xv)
    X_res, y_res = SMOTE(random_state=42).fit_resample(Xt_p, yt)

    m1 = get_mlp(X_res.shape[1]); m2 = get_cnn_gru(X_res.shape[1])
    m1.fit(X_res, y_res, epochs=15, batch_size=32, verbose=0)
    m2.fit(X_res, y_res, epochs=15, batch_size=32, verbose=0)

    p1 = m1.predict(Xv_p).flatten()
    p2 = m2.predict(Xv_p).flatten()

    # Meta-learner for Stacking (trained on current fold's resampled data)
    p1_res = m1.predict(X_res).flatten()
    p2_res = m2.predict(X_res).flatten()
    meta_learner = LogisticRegression().fit(np.column_stack([p1_res, p2_res]), y_res)

    for s in strategies:
        if s == 'Soft Voting': y_prob = (p1 + p2) / 2
        elif s == 'Weighted Voting': y_prob = (0.3 * p1 + 0.7 * p2)
        else: y_prob = meta_learner.predict_proba(np.column_stack([p1, p2]))[:, 1]

        y_pred = (y_prob > 0.5).astype(int)

        cv_results[s]['acc'].append(accuracy_score(yv, y_pred))
        cv_results[s]['f1'].append(f1_score(yv, y_pred))
        cv_results[s]['prec'].append(precision_score(yv, y_pred))
        cv_results[s]['rec'].append(recall_score(yv, y_pred))
        cv_results[s]['auc'].append(roc_auc_score(yv, y_prob))
        cv_results[s]['mcc'].append(matthews_corrcoef(yv, y_pred))
        cv_results[s]['kappa'].append(cohen_kappa_score(yv, y_pred))
        cv_results[s]['brier'].append(brier_score_loss(yv, y_prob))
        cv_results[s]['logloss'].append(log_loss(yv, y_prob))

# ==========================================
# 3. FINAL SUMMARY (MEAN ± STD)
# ==========================================
final_table = []
for s in strategies:
    row = {'Strategy': s}
    for m in metrics_list:
        mean_val = np.mean(cv_results[s][m])
        std_val = np.std(cv_results[s][m])
        row[m] = f"{mean_val:.4f} ± {std_val:.4f}"
    final_table.append(row)

summary_df = pd.DataFrame(final_table)
print("\n" + "="*100)
print("FINAL ENSEMBLE COMPARISON (MEAN ± STD)")
print("="*100)
print(summary_df.to_string(index=False))

Starting 5-Fold Cross-Validation Comparison...
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
483/483 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step

FINAL ENSEMBLE COMPARISON (MEAN ± STD)
       Strategy             acc              f1            prec             rec             auc             